# Binary black hole orbital phase

This notebook demonstrates the model, Kepler solver, phase evolution, and detection of times where $\phi = i\pi$.

The workflow is: evolve the mean anomaly $l(t)$, solve $l = u - e_t\sin(u)$, compute the true anomaly $v$, and then evaluate $\phi = \phi_0 + (1+k)v$.

In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Find src/ from the notebook's current directory.
for root in (Path.cwd(), *Path.cwd().parents):
    if (root / "src" / "oj287_sim" / "main.py").exists():
        sys.path.insert(0, str(root / "src"))
        break
else:
    raise FileNotFoundError("src/oj287_sim/main.py not found")

from oj287_sim.main import simulate_orbit
from oj287_sim.model import BBHModel
from oj287_sim.solver import solve_kepler

FileNotFoundError: src/oj287_sim/main.py not found

In [ ]:
m = BBHModel(
    phi0=0.0,
    k=0.02,
    e_phi=0.3,
    l0=0.0,
    n=1.0,
    n_dot=0.0,
    n_ddot=0.0,
    t0=0.0,
    e_t=0.3,
)

t = np.linspace(0.0, 4.0 * np.pi, 2001)
r = simulate_orbit(m, t)

print(f"samples: {len(r['t'])}")
print(f"crossings: {len(r['phase_crossings'])}")
r["phase_crossings"]

In [ ]:
# One direct solver call for the equation l = u - e*sin(u).
ll = 1.0
uu = solve_kepler(ll, m.e_t)
print(f"l={ll:.3f}, e={m.e_t:.3f}, u={uu:.12f}")
print(f"residual={uu - m.e_t * np.sin(uu) - ll:.3e}")

In [ ]:
cross = r["phase_crossings"]
plt.figure(figsize=(8, 4))
plt.plot(r["t"], r["phi"], label="phi(t)")
plt.scatter(cross, np.interp(cross, r["t"], r["phi"]), color="crimson", s=20, label="phi = i*pi")
plt.xlabel("t")
plt.ylabel("phi")
plt.legend()
plt.tight_layout()
plt.show()

The crossing times are estimated by linear interpolation between neighboring time samples. For higher precision, reduce the time spacing or refine each event with a scalar root finder.